# Sourcemeter Driver Test Notebook

| Layer | Class | Key Methods |
|---|---|---|
| 1 | `Instrument` | `idn()` |
| 2 | `Scpi` | `reset()`, `clear()`, `error()`, `wait()`, `self_test()`, `operation_complete()`, `initialize()` |
| 3 | `Sourcemeter` | Source/Sense config, compliance, output, measurement |

**Instructions:** Run each cell sequentially. Verify the **Expected Result** after each cell.

⚠️ **Safety Note:** The sourcemeter can output voltage and current. Ensure your DUT can handle the configured values before enabling output.

---
## 0. Setup & Connection

In [ ]:
from piec.drivers.autodetect import autodetect

In [ ]:
# --- Option A: Auto-detect ---
smu = autodetect("sourcemeter", verbose=True)

# --- Option B: Manual ---
# from piec.drivers.sourcemeter.keithley2400 import Keithley2400
# smu = Keithley2400("GPIB0::24::INSTR")

✅ **PASS** if no error is raised.

---
## 1. Instrument & SCPI Tests

### 1.1 Identification (`idn`)

In [ ]:
idn_response = smu.idn()
print("IDN Response:")
print(idn_response)

**Expected Result:** `Manufacturer, Model, Serial Number, Firmware Version`

✅ **PASS** if the string matches your physical instrument.

### 1.2 Reset (`reset`)

In [ ]:
smu.reset()
print("Reset command sent.")

**Expected Result:** Sourcemeter returns to factory defaults. Output turns off.

✅ **PASS** if display resets.

### 1.3 Clear, Error, Self Test, Wait, OPC, Initialize

In [ ]:
smu.clear()
print("Clear: OK")

error_response = smu.error()
print("Error:", error_response)

self_test_result = smu.self_test()
print("Self-test:", self_test_result)

smu.wait()
opc = smu.operation_complete()
print("OPC:", opc)

smu.initialize()
print("Initialize: OK")

**Expected Result:** Error: `0`, Self-test: `0`, OPC: `1`

✅ **PASS** if all values match.

---
## 2. Source Configuration Tests

### 2.1 Set Source Function (`set_source_function`)

In [ ]:
smu.set_source_function(channel=1, source_func='VOLT')
print("Source function set to VOLTAGE.")

**Expected Result:** SMU is in **voltage source** mode.

✅ **PASS** if display shows V-Source.

In [ ]:
smu.set_source_function(channel=1, source_func='CURR')
print("Source function set to CURRENT.")

**Expected Result:** SMU is in **current source** mode.

✅ **PASS** if display shows I-Source.

### 2.2 Set Sense Function (`set_sense_function`)

In [ ]:
smu.set_sense_function(channel=1, sense_func='VOLT')
print("Sense function set to VOLT.")

✅ **PASS** if measuring voltage.

In [ ]:
smu.set_sense_function(channel=1, sense_func='CURR')
print("Sense function set to CURR.")

✅ **PASS** if measuring current.

In [ ]:
smu.set_sense_function(channel=1, sense_func='RES')
print("Sense function set to RES.")

✅ **PASS** if measuring resistance.

### 2.3 Set Sense Mode (`set_sense_mode`)

In [ ]:
smu.set_sense_mode(channel=1, sense_mode='2W')
print("Sense mode: 2-wire.")

✅ **PASS** if 2-wire mode is active.

In [ ]:
smu.set_sense_mode(channel=1, sense_mode='4W')
print("Sense mode: 4-wire.")

✅ **PASS** if 4-wire indicator is shown.

### 2.4 Set Source Voltage (`set_source_voltage`)

In [ ]:
smu.set_source_function(channel=1, source_func='VOLT')
smu.set_source_voltage(channel=1, voltage=0.0)
print("Source voltage set to 0 V.")

**Expected Result:** Voltage source level reads **0 V**.

✅ **PASS** if voltage level matches.

In [ ]:
smu.set_source_voltage(channel=1, voltage=1.0)
print("Source voltage set to 1 V.")

✅ **PASS** if source voltage reads 1 V.

### 2.5 Set Source Current (`set_source_current`)

In [ ]:
smu.set_source_function(channel=1, source_func='CURR')
smu.set_source_current(channel=1, current=0.001)
print("Source current set to 1 mA.")

**Expected Result:** Current source level reads **1 mA**.

✅ **PASS** if current level matches.

### 2.6 Set Compliance

In [ ]:
# Voltage compliance (used in current source mode)
smu.set_voltage_compliance(channel=1, voltage_compliance=10)
print("Voltage compliance set to 10 V.")

✅ **PASS** if voltage compliance reads 10 V.

In [ ]:
# Current compliance (used in voltage source mode)
smu.set_source_function(channel=1, source_func='VOLT')
smu.set_current_compliance(channel=1, current_compliance=0.1)
print("Current compliance set to 100 mA.")

✅ **PASS** if current compliance reads 100 mA.

---
## 3. Convenience Configuration Tests

### 3.1 Configure Voltage Source (`configure_voltage_source`)

In [ ]:
smu.configure_voltage_source(
    channel=1,
    voltage=0.5,
    current_compliance=0.01
)
print("Configured: V-Source 0.5 V, I-compliance 10 mA.")

**Expected Result:**
- Source: Voltage, 0.5 V
- I-compliance: 10 mA

✅ **PASS** if all match.

### 3.2 Configure Current Source (`configure_current_source`)

In [ ]:
smu.configure_current_source(
    channel=1,
    current=0.001,
    voltage_compliance=5
)
print("Configured: I-Source 1 mA, V-compliance 5 V.")

**Expected Result:**
- Source: Current, 1 mA
- V-compliance: 5 V

✅ **PASS** if all match.

---
## 4. Output & Measurement Tests

### 4.1 Output On/Off (`output`)

In [ ]:
# Start with a safe voltage source config
smu.configure_voltage_source(channel=1, voltage=0.0, current_compliance=0.01)

smu.output(channel=1, on=True)
print("Output ON.")

**Expected Result:** Output indicator illuminates. Sourcing 0 V.

✅ **PASS** if output is ON.

In [ ]:
smu.output(channel=1, on=False)
print("Output OFF.")

✅ **PASS** if output turns OFF.

### 4.2 Quick Read (`quick_read`)

In [ ]:
smu.configure_voltage_source(channel=1, voltage=0.0, current_compliance=0.01)
smu.output(channel=1, on=True)

reading = smu.quick_read(channel=1)
print(f"Quick Read: {reading}")

smu.output(channel=1, on=False)

**Expected Result:** A numeric value for the configured sense function.

✅ **PASS** if a reasonable value is returned.

### 4.3 Get Voltage (`get_voltage`)

In [ ]:
smu.configure_voltage_source(channel=1, voltage=1.0, current_compliance=0.01)
smu.output(channel=1, on=True)

measured_v = smu.get_voltage(channel=1)
print(f"Measured voltage: {measured_v} V")

smu.output(channel=1, on=False)

**Expected Result:** With open terminals sourcing 1 V, measured voltage should be approximately **1.0 V**.

✅ **PASS** if voltage ≈ 1.0 V.

### 4.4 Get Current (`get_current`)

In [ ]:
smu.configure_voltage_source(channel=1, voltage=0.0, current_compliance=0.01)
smu.output(channel=1, on=True)

measured_i = smu.get_current(channel=1)
print(f"Measured current: {measured_i} A")

smu.output(channel=1, on=False)

**Expected Result:** With open terminals, current should be near **0 A**.

✅ **PASS** if current ≈ 0 A.

---
## 5. Cleanup

In [ ]:
smu.output(channel=1, on=False)
smu.reset()
print("Output OFF. SMU reset.")

---
## Test Summary

| # | Method(s) Tested | Pass/Fail |
|---|------------------|-----------|
| 0 | `__init__` / `autodetect` | |
| 1.1 | `idn()` | |
| 1.2 | `reset()` | |
| 1.3 | `clear()`, `error()`, `self_test()`, `wait()`, `operation_complete()`, `initialize()` | |
| 2.1 | `set_source_function()` | |
| 2.2 | `set_sense_function()` | |
| 2.3 | `set_sense_mode()` | |
| 2.4 | `set_source_voltage()` | |
| 2.5 | `set_source_current()` | |
| 2.6 | `set_voltage_compliance()`, `set_current_compliance()` | |
| 3.1 | `configure_voltage_source()` | |
| 3.2 | `configure_current_source()` | |
| 4.1 | `output()` | |
| 4.2 | `quick_read()` | |
| 4.3 | `get_voltage()` | |
| 4.4 | `get_current()` | |
| 5 | `output(off)`, `reset()` | |

**Technician Signature:** _______________  
**Date:** _______________  
**Instrument Serial #:** _______________